# Jewelry Image Generation with FLUX.2 [klein] 9B Base + LoRA

**Google Colab research notebook** for the user's `design_dataset`.

### Dataset expected in Google Drive
```text
design_dataset/
├── enhanced_metadata.csv
└── images/
    ├── train/
    │   └── ...
    ├── validation/
    │   └── ...
    └── test/
        └── ...
```

### Architecture
```text
Jewelry images + enhanced metadata
                ↓
Canonical structured prompt
                +
Narrative caption
                ↓
FLUX.2 [klein] 9B Base
                ↓
PEFT LoRA
                ↓
Jewelry-specialized generator
                ↓
Same-prompt / same-seed benchmark
       ┌────────┴────────┐
       ↓                 ↓
Semantic alignment   Attribute accuracy
       └────────┬────────┘
                ↓
          Final evaluation
```

This notebook deliberately uses **Klein 9B Base** for the main training experiment rather than FLUX.2 [dev] 32B. The 9B Base checkpoint is the undistilled variant intended for fine-tuning/LoRA/research, while the 32B model has much higher memory requirements.

### Research evaluation note
The notebook uses the official Hugging Face Klein LoRA trainer. It enables the trainer's validation-image generation and then performs fixed, held-out checkpoint evaluation with SigLIP and optional VLM attribute scoring. The official trainer does not provide a native validation diffusion-loss loop, so no synthetic validation-loss number is reported.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## What is improved over the SD 1.5 experiment

- FLUX.2 Klein 9B Base instead of Stable Diffusion 1.5.
- Official Hugging Face Diffusers Klein LoRA trainer.
- Uses the CSV `split` column; no random train/test mixing.
- Uses **all appropriate metadata fields** in a canonical prompt:
  `category`, `gemstone`, `gemstone_color`, `gemstone_cut`, `weight_carats`, `metal`, `setting`, `style`, `occasion`, `caption_tags`, `caption_narrative`.
- Keeps `image_id`, `image_path`, `split`, and `source_file` out of the prompt.
- Aspect-ratio bucketing instead of forcing every jewelry photo into a square crop.
- VAE latent caching, gradient checkpointing, CPU offload, 8-bit AdamW.
- Deterministic base-vs-fine-tuned comparison using the **same seeds**.
- Optional SigLIP2 prompt alignment.
- Optional Qwen3-VL attribute scoring.
- Saves all prepared metadata, LoRA weights, prompts, generated images, and metrics back to Google Drive.

In [ ]:
# ============================================================
# 1. INSTALL CURRENT DIFFUSERS + TRAINING DEPENDENCIES
# ============================================================

!pip -q install -U \
    "huggingface_hub>=1.32.0,<2.0" \
    accelerate \
    transformers \
    peft \
    bitsandbytes \
    datasets \
    sentencepiece \
    safetensors \
    pillow \
    matplotlib \
    tqdm \
    torchmetrics

!pip install -q "pandas==2.2.3"

# Clean previous Diffusers checkout
!rm -rf /content/diffusers

# Get current Diffusers source
!git clone -q --depth 1 https://github.com/huggingface/diffusers.git /content/diffusers

%cd /content/diffusers

# Install Diffusers + its current dependencies
!pip -q install -e .

# Install FLUX DreamBooth requirements
%cd /content/diffusers/examples/dreambooth
!pip -q install -r requirements_flux.txt

%cd /content

print("Environment installation completed.")

In [ ]:
# TorchAO is intentionally NOT installed for this A100 40 GB experiment.
# FP8 training is disabled. Installing TorchAO here is unnecessary and can introduce
# version conflicts with the PyTorch environment.
print("TorchAO installation skipped: official FP8 training is disabled for A100/Ampere.")


In [ ]:
# No additional TorchAO installation is required.
print("Environment will use BF16 + the official Diffusers Klein trainer.")


In [ ]:
# ============================================================
# 2. IMPORTS + GOOGLE DRIVE
# ============================================================
import os, re, gc, json, math, random, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")

In [ ]:
# ============================================================
# 3. EXPERIMENT CONFIG
# ============================================================
SEED = 42

DATASET_ROOT = Path("/content/drive/MyDrive/design_dataset")
METADATA_CSV = DATASET_ROOT / "enhanced_metadata.csv"
IMAGES_ROOT = DATASET_ROOT / "images"

DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/flux2_jewelry_experiment")
LOCAL_DATASET_ROOT = Path("/content/flux2_jewelry_train")

MODEL_ID = "black-forest-labs/FLUX.2-klein-base-9B"

# A100 40 GB target. Keep 1024px for the research experiment.
TRAIN_RESOLUTION = 1024
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4

# LoRA
LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Optimization
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-4
WARMUP_STEPS = 100
CAPTION_DROPOUT = 0.05
TARGET_EPOCHS = 4
MIN_TRAIN_STEPS = 1200
MAX_TRAIN_STEPS = 3000

# Recovery / validation
CHECKPOINT_STEPS = 500
CHECKPOINTS_TO_KEEP = 5
RESUME_FROM_CHECKPOINT = "auto"
VALIDATION_EPOCHS = 1
NUM_VALIDATION_IMAGES = 2

# A100 40 GB memory configuration.
# IMPORTANT: do NOT enable FP8 on A100/Ampere for this experiment.
FP8_TRAINING = False
GRADIENT_CHECKPOINTING = True
CACHE_LATENTS = True
OFFLOAD = True
USE_8BIT_ADAM = True
ALLOW_TF32 = True

# The dataset captions are short. 128 gives enough room for them while
# reducing Qwen3 prompt-embedding cache memory on the 40 GB A100.
MAX_SEQUENCE_LENGTH = 128

# The official Diffusers trainer is used unchanged.
# There is deliberately NO custom persistent cache and NO trainer monkey patch.

BENCHMARK_COUNT = 20
INFERENCE_STEPS = 50
GUIDANCE_SCALE = 4.0
LORA_SCALE = 1.0

RUN_SIGLIP = True
RUN_VLM_ATTRIBUTE_EVAL = True
RUN_FID = False
VLM_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
SIGLIP_MODEL_ID = "google/siglip2-base-patch16-512"

random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# ============================================================
# 4. GPU CHECK
# ============================================================
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select an NVIDIA GPU runtime in Colab.")

gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
major, minor = torch.cuda.get_device_capability(0)
MIXED_PRECISION = "bf16" if major >= 8 else "fp16"

print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_vram_gb:.1f} GB")
print(f"Compute capability: {major}.{minor}")
print(f"Mixed precision: {MIXED_PRECISION}")

if "A100" not in gpu_name or gpu_vram_gb < 38:
    raise RuntimeError(
        f"This notebook is configured specifically for an NVIDIA A100 40 GB-class GPU. "
        f"Detected: {gpu_name} ({gpu_vram_gb:.1f} GB)."
    )

if MIXED_PRECISION != "bf16":
    raise RuntimeError("A100 should use BF16 for this experiment.")


## Hugging Face authentication

The first model download may require Hugging Face authentication/accepted access terms. Run the next cell once per Colab session.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ============================================================
# 5. LOAD + VALIDATE CSV
# ============================================================
if not DATASET_ROOT.exists():
    raise FileNotFoundError(DATASET_ROOT)
if not METADATA_CSV.exists():
    raise FileNotFoundError(METADATA_CSV)
if not IMAGES_ROOT.exists():
    raise FileNotFoundError(IMAGES_ROOT)

df = pd.read_csv(METADATA_CSV)

required = [
    "image_id","image_path","split","category","gemstone","gemstone_color",
    "gemstone_cut","weight_carats","metal","setting","style","occasion",
    "caption_tags","caption_narrative","source_file"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing CSV columns: {missing}")

print("Metadata shape:", df.shape)
print("Columns:", list(df.columns))
print("\nSplit counts:")
print(df["split"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 6. NORMALIZE METADATA
# ============================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).strip())

for c in required:
    if c != "weight_carats":
        df[c] = df[c].map(clean_text)

df["weight_carats_numeric"] = pd.to_numeric(df["weight_carats"], errors="coerce")
df["split_normalized"] = (
    df["split"].str.lower().str.strip()
    .replace({"val":"validation","valid":"validation","training":"train","testing":"test"})
)

bad = sorted(set(df["split_normalized"]) - {"train","validation","test"})
if bad:
    print("Unexpected split values:", bad)

print(df["split_normalized"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 7. RESOLVE IMAGE PATHS ROBUSTLY
# ============================================================
exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
all_images = [p for p in IMAGES_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in exts]
by_name, by_stem = {}, {}
for p in all_images:
    by_name.setdefault(p.name.lower(), []).append(p)
    by_stem.setdefault(p.stem.lower(), []).append(p)

def resolve_path(row):
    raw = clean_text(row["image_path"]).replace("\\", "/")
    candidates = []
    if raw:
        rp = Path(raw)
        if rp.is_absolute():
            candidates.append(rp)
        candidates += [DATASET_ROOT / raw, IMAGES_ROOT / raw]
        parts = raw.split("/")
        if "images" in parts:
            i = parts.index("images")
            candidates.append(IMAGES_ROOT.joinpath(*parts[i+1:]))
    for p in candidates:
        if p.exists() and p.is_file():
            return p
    image_id = clean_text(row["image_id"])
    hits = by_name.get(Path(image_id).name.lower(), [])
    if len(hits) == 1:
        return hits[0]
    hits = by_stem.get(Path(image_id).stem.lower(), [])
    if len(hits) == 1:
        return hits[0]
    return None

df["resolved_path"] = df.apply(resolve_path, axis=1)
missing_rows = df[df["resolved_path"].isna()]
print("Resolved:", int(df["resolved_path"].notna().sum()))
print("Unresolved:", len(missing_rows))
if len(missing_rows):
    display(missing_rows[["image_id","image_path","split","category"]].head(30))
    raise RuntimeError("Unresolved image paths found. Fix the dataset before training.")

In [ ]:
# ============================================================
# 8. VERIFY IMAGE INTEGRITY
# ============================================================
bad_images = []
for i, row in df.iterrows():
    p = Path(row["resolved_path"])
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        bad_images.append({"index":i,"image_id":row["image_id"],"path":str(p),"error":repr(e)})

print("Corrupt/unreadable images:", len(bad_images))
if bad_images:
    display(pd.DataFrame(bad_images).head(30))
    raise RuntimeError("Corrupt or unreadable images found. Do not replace them with blank images.")

In [ ]:
# ============================================================
# 9. BUILD CANONICAL JEWELRY PROMPT
# ============================================================
def weight_text(x):
    if pd.isna(x) or str(x).strip() == "":
        return ""
    try:
        return f"{float(x):g} carats"
    except Exception:
        return clean_text(x)

def build_prompt(row):
    fields = []
    for label, col in [
        ("Category", "category"),
        ("Gemstone", "gemstone"),
        ("Gemstone color", "gemstone_color"),
        ("Gemstone cut", "gemstone_cut"),
        ("Approximate gemstone weight", "weight_carats"),
        ("Metal", "metal"),
        ("Setting", "setting"),
        ("Style", "style"),
        ("Occasion", "occasion"),
        ("Tags", "caption_tags"),
    ]:
        val = weight_text(row[col]) if col == "weight_carats" else clean_text(row[col])
        if val:
            suffix = val if col == "weight_carats" else val
            fields.append(f"{label}: {suffix}.")

    prompt = (
        "A high-end jewelry product photograph. Faithfully represent these specifications: "
        + " ".join(fields)
    )

    narrative = clean_text(row["caption_narrative"])
    if narrative:
        prompt += " Visual description: " + narrative

    prompt += (
        " Show the complete jewelry item clearly. Preserve accurate jewelry geometry, "
        "realistic gemstone appearance, physically plausible metal, fine craftsmanship, "
        "premium commercial product photography, sharp subject, realistic materials and reflections."
    )
    return prompt

df["structured_prompt"] = df.apply(build_prompt, axis=1)
print(df[["image_id","structured_prompt"]].head(3).to_string(index=False))

In [ ]:
# ============================================================
# 10. SPLIT TABLES + DATASET REPORT
# ============================================================
train_df = df[df["split_normalized"] == "train"].copy()
val_df = df[df["split_normalized"] == "validation"].copy()
test_df = df[df["split_normalized"] == "test"].copy()

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))

if len(train_df) == 0 or len(test_df) == 0:
    raise RuntimeError("Training and held-out test data are both required.")

print("\nTrain category counts:")
print(train_df["category"].value_counts().head(30))
print("\nTrain gemstone counts:")
print(train_df["gemstone"].value_counts().head(30))
print("\nTrain metal counts:")
print(train_df["metal"].value_counts().head(30))

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
train_df.to_csv(DRIVE_OUTPUT_ROOT / "train_metadata_prepared.csv", index=False)
val_df.to_csv(DRIVE_OUTPUT_ROOT / "validation_metadata_prepared.csv", index=False)
test_df.to_csv(DRIVE_OUTPUT_ROOT / "test_metadata_prepared.csv", index=False)

In [ ]:
# ============================================================
# 11. VISUAL SAMPLE
# ============================================================
def show_grid(frame, n=12):
    sample = frame.sample(min(n, len(frame)), random_state=SEED).reset_index(drop=True)
    cols = 4
    rows = math.ceil(len(sample)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4*rows))
    axes = np.array(axes).reshape(-1)
    for ax, (_, row) in zip(axes, sample.iterrows()):
        ax.imshow(Image.open(row["resolved_path"]).convert("RGB"))
        ax.set_title(f"{row['category']} | {row['gemstone']} | {row['metal']}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(sample):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_grid(train_df)

## Dataset diagnostics for the research record

These diagnostics document the training data before the long training run. They are useful for identifying class imbalance, missing metadata, unusual image dimensions, and prompt-length variation.

In [ ]:
# ============================================================
# 11A. DATASET STATISTICS + DISTRIBUTION VISUALIZATION
# ============================================================
# Research-oriented diagnostics. These do not modify the dataset.

def _barplot_counts(series, title, xlabel="Count", top_n=20, rotate=35):
    counts = series.fillna("").replace("", "Unknown").value_counts().head(top_n)
    fig, ax = plt.subplots(figsize=(10, 5))
    counts.sort_values().plot(kind="barh", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()
    return counts

# Split distribution
split_counts = df["split_normalized"].value_counts().reindex(
    ["train", "validation", "test"], fill_value=0
)
fig, ax = plt.subplots(figsize=(8, 4))
split_counts.plot(kind="bar", ax=ax)
ax.set_title("Dataset split distribution")
ax.set_xlabel("Split")
ax.set_ylabel("Images")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
fig.savefig(DRIVE_OUTPUT_ROOT / "dataset_split_distribution.png", dpi=160)
plt.show()

# Main categorical distributions
for col, label in [
    ("category", "Jewelry category distribution"),
    ("gemstone", "Gemstone distribution"),
    ("metal", "Metal distribution"),
]:
    counts = _barplot_counts(train_df[col], label, top_n=20)
    counts.to_csv(DRIVE_OUTPUT_ROOT / f"{col}_distribution.csv")

# Metadata completeness
metadata_cols = [
    "category", "gemstone", "gemstone_color", "gemstone_cut",
    "weight_carats", "metal", "setting", "style", "occasion",
    "caption_tags", "caption_narrative"
]
completeness = []
for col in metadata_cols:
    nonempty = train_df[col].map(clean_text).ne("").mean() * 100
    completeness.append({"field": col, "filled_percent": nonempty})
completeness_df = pd.DataFrame(completeness).sort_values("filled_percent")

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(completeness_df["field"], completeness_df["filled_percent"])
ax.set_xlim(0, 100)
ax.set_xlabel("Filled metadata (%)")
ax.set_title("Training metadata completeness")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
fig.savefig(DRIVE_OUTPUT_ROOT / "metadata_completeness.png", dpi=160)
plt.show()

display(completeness_df)

# Prompt length distribution
train_df = train_df.copy()
train_df["prompt_chars"] = train_df["structured_prompt"].map(len)
train_df["prompt_words"] = train_df["structured_prompt"].map(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_df["prompt_words"], bins=30)
ax.set_xlabel("Prompt length (words)")
ax.set_ylabel("Images")
ax.set_title("Training prompt-length distribution")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
fig.savefig(DRIVE_OUTPUT_ROOT / "prompt_length_distribution.png", dpi=160)
plt.show()

print(
    f"Prompt words — mean: {train_df['prompt_words'].mean():.1f}, "
    f"median: {train_df['prompt_words'].median():.0f}, "
    f"min: {train_df['prompt_words'].min()}, max: {train_df['prompt_words'].max()}"
)

# Image dimensions / aspect ratios
dimension_rows = []
for p in train_df["resolved_path"]:
    try:
        with Image.open(p) as im:
            w, h = im.size
        dimension_rows.append({"width": w, "height": h, "aspect_ratio": w / h})
    except Exception:
        pass

dimensions_df = pd.DataFrame(dimension_rows)
if len(dimensions_df):
    display(dimensions_df.describe().round(3))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(dimensions_df["aspect_ratio"], bins=30)
    ax.axvline(1.0, linestyle="--", label="Square")
    ax.set_xlabel("Width / height")
    ax.set_ylabel("Images")
    ax.set_title("Training image aspect-ratio distribution")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    fig.savefig(DRIVE_OUTPUT_ROOT / "image_aspect_ratio_distribution.png", dpi=160)
    plt.show()

# Save a compact diagnostic table
dataset_diagnostics = {
    "total_images": int(len(df)),
    "train_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "unique_categories": int(train_df["category"].nunique()),
    "unique_gemstones": int(train_df["gemstone"].nunique()),
    "unique_metals": int(train_df["metal"].nunique()),
    "mean_prompt_words": float(train_df["prompt_words"].mean()),
    "median_prompt_words": float(train_df["prompt_words"].median()),
}
with (DRIVE_OUTPUT_ROOT / "dataset_diagnostics.json").open("w", encoding="utf-8") as f:
    json.dump(dataset_diagnostics, f, indent=2)

print(json.dumps(dataset_diagnostics, indent=2))


## Stage 1: stage the train split locally

Google Drive is good for storage but is not ideal as the live training filesystem. The next cell copies only the training images into Colab local storage and creates a Hugging Face `imagefolder` dataset with `metadata.jsonl`.

In [ ]:
# ============================================================
# 12. COPY TRAIN IMAGES + CREATE metadata.jsonl
# ============================================================
if LOCAL_DATASET_ROOT.exists():
    shutil.rmtree(LOCAL_DATASET_ROOT)
LOCAL_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

records = []
for i, (_, row) in enumerate(train_df.iterrows()):
    src = Path(row["resolved_path"])
    safe_id = re.sub(r"[^A-Za-z0-9._-]+", "_", row["image_id"] or src.stem)
    name = f"{i:06d}_{safe_id}{src.suffix.lower()}"
    dst = LOCAL_DATASET_ROOT / name
    shutil.copy2(src, dst)
    records.append({
        "file_name": name,
        "text": row["structured_prompt"],
        "image_id": row["image_id"],
    })

with (LOCAL_DATASET_ROOT / "metadata.jsonl").open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Local dataset:", LOCAL_DATASET_ROOT)
print("Images copied:", len(records))

In [ ]:
# ============================================================
# 13. TEST HUGGING FACE IMAGEFOLDER LOADING
# ============================================================
from datasets import load_dataset
hf_ds = load_dataset("imagefolder", data_dir=str(LOCAL_DATASET_ROOT), split="train")
print(hf_ds)
print(hf_ds.column_names)
print("Example caption:\n", hf_ds[0]["text"])

In [ ]:
# ============================================================
# 14. CALCULATE MAX TRAINING STEPS
# ============================================================
steps_per_epoch = math.ceil(len(train_df) / TRAIN_BATCH_SIZE / GRAD_ACCUM_STEPS)
raw_steps = steps_per_epoch * TARGET_EPOCHS
MAX_TRAIN_STEPS_FINAL = max(MIN_TRAIN_STEPS, min(MAX_TRAIN_STEPS, raw_steps))

print("Images:", len(train_df))
print("Optimizer steps/epoch:", steps_per_epoch)
print("Target epochs:", TARGET_EPOCHS)
print("Max train steps:", MAX_TRAIN_STEPS_FINAL)

In [ ]:
# ============================================================
# 15. SAVE EXPERIMENT CONFIG
# ============================================================
config = {
    "seed": SEED,
    "model_id": MODEL_ID,
    "train_resolution": TRAIN_RESOLUTION,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "caption_dropout": CAPTION_DROPOUT,
    "max_train_steps": MAX_TRAIN_STEPS_FINAL,
    "mixed_precision": MIXED_PRECISION,
    "fp8_training": FP8_TRAINING,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "cache_latents": CACHE_LATENTS,
    "offload": OFFLOAD,
    "use_8bit_adam": USE_8BIT_ADAM,
    "allow_tf32": ALLOW_TF32,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "checkpoint_steps": CHECKPOINT_STEPS,
    "checkpoints_to_keep": CHECKPOINTS_TO_KEEP,
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
    "persistent_cache": False,
    "trainer_source_patched": False,
}
with (DRIVE_OUTPUT_ROOT / "experiment_config.json").open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))


## Pre-training configuration

This notebook uses the **official Diffusers FLUX.2 Klein LoRA trainer without modifying its source code**.

For the A100 40 GB target:
- 1024px training resolution
- batch size 1
- gradient accumulation 4
- BF16
- gradient checkpointing
- VAE latent caching
- CPU offload for VAE/text encoder
- 8-bit AdamW
- TF32
- maximum prompt sequence length 128

The previous custom per-image Google Drive cache has intentionally been removed. The official trainer's cache is kept in memory during its precomputation stage. Official training checkpoints are still written to Drive and can be used with `--resume_from_checkpoint latest` after a checkpoint has been created.


In [ ]:
# ============================================================
# 15B. GOOGLE DRIVE STORAGE PREFLIGHT
# ============================================================
drive_usage = shutil.disk_usage("/content/drive")
drive_free_gb = drive_usage.free / (1024**3)

print(f"Google Drive free space visible to Colab: {drive_free_gb:.2f} GB")
print(f"Experiment output directory: {DRIVE_OUTPUT_ROOT}")
print(f"Checkpoint policy: every {CHECKPOINT_STEPS} steps; keep {CHECKPOINTS_TO_KEEP}")

if drive_free_gb < 1.0:
    raise RuntimeError(
        "Less than 1 GB of Drive space is available. "
        "Free space before starting the long training run."
    )

print("Storage preflight: OK")


## Stage 2: LoRA training

This stage uses the official Hugging Face Diffusers `train_dreambooth_lora_flux2_klein.py` script **unchanged**.

Memory/reliability configuration:

- FLUX.2 Klein 9B Base
- 1024px target resolution with aspect-ratio buckets
- LoRA rank 16 / alpha 16
- 5e-5 learning rate
- BF16 on A100
- FP8 disabled
- gradient checkpointing
- official VAE latent/prompt precomputation
- CPU offload for VAE/text encoder
- 8-bit AdamW
- batch 1 + gradient accumulation 4
- TF32 enabled on Ampere
- Qwen3 maximum sequence length 128

### Recovery

The official trainer saves complete training-state checkpoints every 500 optimization steps and supports `--resume_from_checkpoint latest`. No internal trainer source is modified, and no custom cache argument is injected.


In [ ]:
# ============================================================
# 16. BUILD + VALIDATE OFFICIAL DIFFUSERS TRAINING COMMAND
# ============================================================
# IMPORTANT: this cell NEVER edits train_dreambooth_lora_flux2_klein.py.
# It only builds a command using arguments supported by the official trainer.

TRAIN_SCRIPT = Path("/content/diffusers/examples/dreambooth/train_dreambooth_lora_flux2_klein.py")
if not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(f"Official Klein trainer not found: {TRAIN_SCRIPT}")

OUTPUT_DIR = DRIVE_OUTPUT_ROOT / "lora_training"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "accelerate", "launch", "--num_processes=1",
    f"--mixed_precision={MIXED_PRECISION}",
    str(TRAIN_SCRIPT),
    "--pretrained_model_name_or_path", MODEL_ID,
    "--dataset_name", str(LOCAL_DATASET_ROOT),
    "--image_column", "image",
    "--caption_column", "text",
    "--output_dir", str(OUTPUT_DIR),
    "--instance_prompt", "a high-end jewelry product photograph",
    "--resolution", str(TRAIN_RESOLUTION),
    "--use_aspect_ratio_buckets",
    "--train_batch_size", str(TRAIN_BATCH_SIZE),
    "--gradient_accumulation_steps", str(GRAD_ACCUM_STEPS),
    "--learning_rate", str(LEARNING_RATE),
    "--adam_weight_decay", str(WEIGHT_DECAY),
    "--lr_scheduler", "constant_with_warmup",
    "--lr_warmup_steps", str(WARMUP_STEPS),
    "--rank", str(LORA_RANK),
    "--lora_alpha", str(LORA_ALPHA),
    "--lora_dropout", str(LORA_DROPOUT),
    "--caption_dropout", str(CAPTION_DROPOUT),
    "--max_train_steps", str(MAX_TRAIN_STEPS_FINAL),
    "--checkpointing_steps", str(CHECKPOINT_STEPS),
    "--checkpoints_total_limit", str(CHECKPOINTS_TO_KEEP),
    "--guidance_scale", "1.0",
    "--validation_prompt", "A professional high-end studio product photograph of a luxury sapphire jewelry piece, extremely detailed gemstone and metal, clean neutral background",
    "--validation_epochs", str(VALIDATION_EPOCHS),
    "--num_validation_images", str(NUM_VALIDATION_IMAGES),
    "--seed", str(SEED),
    "--max_sequence_length", str(MAX_SEQUENCE_LENGTH),
    "--report_to", "tensorboard",
    "--dataloader_num_workers", "0",
    "--skip_final_inference",
]

if GRADIENT_CHECKPOINTING:
    cmd += ["--gradient_checkpointing"]
if CACHE_LATENTS:
    cmd += ["--cache_latents"]
if OFFLOAD:
    cmd += ["--offload"]
if USE_8BIT_ADAM:
    cmd += ["--use_8bit_adam"]
if ALLOW_TF32:
    cmd += ["--allow_tf32"]
if FP8_TRAINING:
    raise RuntimeError("FP8_TRAINING must remain False for the A100 40 GB configuration.")

# Auto-resume only from an official training-state checkpoint.
if RESUME_FROM_CHECKPOINT == "auto":
    checkpoint_dirs = sorted(
        OUTPUT_DIR.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )
    if checkpoint_dirs:
        cmd += ["--resume_from_checkpoint", "latest"]
        print(f"Auto-resume: {checkpoint_dirs[-1].name}")
    else:
        print("Auto-resume: no checkpoint found; starting from step 0.")
elif RESUME_FROM_CHECKPOINT is not None:
    cmd += ["--resume_from_checkpoint", str(RESUME_FROM_CHECKPOINT)]

print("Training configuration:")
print(f"  GPU target           = A100 40 GB")
print(f"  resolution           = {TRAIN_RESOLUTION}")
print(f"  mixed precision      = {MIXED_PRECISION}")
print(f"  FP8 training         = {FP8_TRAINING}")
print(f"  max sequence length  = {MAX_SEQUENCE_LENGTH}")
print(f"  batch size           = {TRAIN_BATCH_SIZE}")
print(f"  grad accumulation    = {GRAD_ACCUM_STEPS}")
print(f"  max train steps      = {MAX_TRAIN_STEPS_FINAL}")
print(f"  checkpoint interval  = {CHECKPOINT_STEPS}")
print(f"  output directory     = {OUTPUT_DIR}")
print("\nTraining command:")
print(" ".join(map(str, cmd)))

# Interface preflight: ask the official trainer to print its help and verify
# that the arguments used above are accepted. This does not start training.
help_result = subprocess.run(
    ["accelerate", "launch", "--num_processes=1", str(TRAIN_SCRIPT), "--help"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False,
)
if help_result.returncode != 0:
    print(help_result.stdout[-12000:])
    raise RuntimeError("Official Klein trainer --help failed. Fix the environment before starting training.")

required_flags = [
    "--max_sequence_length", "--checkpointing_steps", "--resume_from_checkpoint",
    "--gradient_checkpointing", "--cache_latents", "--offload", "--use_8bit_adam",
    "--validation_prompt", "--validation_epochs", "--num_validation_images",
    "--use_aspect_ratio_buckets", "--skip_final_inference",
]
missing_flags = [flag for flag in required_flags if flag not in help_result.stdout]
if missing_flags:
    raise RuntimeError(f"Official trainer is missing expected arguments: {missing_flags}")
print("\nOfficial trainer interface preflight: PASSED")


In [ ]:
from huggingface_hub import model_info

MODEL_ID = "black-forest-labs/FLUX.2-klein-base-9B"

info = model_info(MODEL_ID)

print("Model:", info.id)
print("Revision:", info.sha)

In [ ]:
# ============================================================
# 17. RUN TRAINING — LIVE OUTPUT + DRIVE LOG
# ============================================================
# This cell streams stdout/stderr to the Colab output AND saves the same
# lines to Drive. Do not use subprocess.run(... stdout=log_file), because
# that would hide trainer output until the entire multi-hour job finishes.

import subprocess, os

env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

TRAIN_LOG = DRIVE_OUTPUT_ROOT / "training_console.log"
TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)

print("Starting training...")
print("Live trainer output will appear below and will also be saved to:", TRAIN_LOG)
print("=" * 80)

with TRAIN_LOG.open("a", encoding="utf-8", buffering=1) as log_file:
    log_file.write("\n\n===== NEW TRAINING INVOCATION =====\n")
    log_file.write("COMMAND: " + " ".join(map(str, cmd)) + "\n\n")
    log_file.flush()

    process = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()
    except KeyboardInterrupt:
        print("\n\nTraining interrupted by user. The process will be terminated.")
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        if process.stdout is not None:
            process.stdout.close()

    return_code = process.wait()

print("=" * 80)
if return_code != 0:
    raise RuntimeError(
        f"Training failed with exit code {return_code}. Full log: {TRAIN_LOG}"
    )

print("Training completed successfully.")
print(f"Training log: {TRAIN_LOG}")


## Training log / checkpoint diagnostics

The training cell below streams output live to Colab **and** writes the same output to Drive. If training finishes or fails, this cell can be run afterward to inspect the persisted log.


In [ ]:
# ============================================================
# 17A. OFFICIAL CHECKPOINT + LOG STATUS
# ============================================================
TRAIN_LOG = DRIVE_OUTPUT_ROOT / "training_console.log"
checkpoint_dirs = sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
)

print(f"Training log exists: {TRAIN_LOG.exists()}")
if TRAIN_LOG.exists():
    print("\n--- latest training log ---")
    print(TRAIN_LOG.read_text(encoding="utf-8", errors="ignore")[-16000:])

print("\nOfficial training checkpoints:")
for p in checkpoint_dirs:
    print(" -", p.name)
if not checkpoint_dirs:
    print(" - none yet")


In [ ]:
!pip show torch torchao diffusers accelerate transformers peft

## Stage 2B: research checkpoint and validation monitoring

The official Hugging Face Klein LoRA trainer supports:
- training-loss logging to TensorBoard;
- checkpoint saving/resume;
- validation **image generation** during training.

It does **not** implement a held-out diffusion validation-loss loop. Therefore this notebook deliberately does not fabricate a `val_loss` column.

Instead, the research record combines:
1. training loss from TensorBoard;
2. validation images generated every epoch by the official trainer;
3. retained training-state checkpoints for recovery;
4. final held-out test metrics (SigLIP + optional VLM attribute accuracy);
5. fixed-seed base-vs-fine-tuned comparisons.

This is safer and faithful to the official trainer implementation.


In [ ]:
# ============================================================
# 18. CHECKPOINT INVENTORY + STORAGE REPORT
# ============================================================
checkpoint_dirs = sorted(
    [p for p in OUTPUT_DIR.glob("checkpoint-*") if p.is_dir()],
    key=lambda p: int(re.search(r"checkpoint-(\d+)", p.name).group(1))
    if re.search(r"checkpoint-(\d+)", p.name) else -1
)

rows = []
for p in checkpoint_dirs:
    size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / (1024**3)
    m = re.search(r"checkpoint-(\d+)", p.name)
    rows.append({
        "checkpoint": p.name,
        "step": int(m.group(1)) if m else -1,
        "size_gb": round(size_gb, 3),
        "path": str(p),
    })

checkpoint_inventory_df = pd.DataFrame(rows)
checkpoint_inventory_df.to_csv(
    DRIVE_OUTPUT_ROOT / "checkpoint_inventory.csv", index=False
)

display(checkpoint_inventory_df)
print(f"Total checkpoint storage: {checkpoint_inventory_df['size_gb'].sum():.3f} GB"
      if len(checkpoint_inventory_df) else "No checkpoints found.")


In [ ]:
# ============================================================
# 19. READ TRAINING LOSS FROM TENSORBOARD EVENT FILES
# ============================================================
# This extracts the actual scalar values logged by the official trainer.
# It is the TRAINING loss, not validation loss.

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

event_files = list(OUTPUT_DIR.rglob("events.out.tfevents.*"))
print("TensorBoard event files:", len(event_files))

tb_rows = []
for event_file in event_files:
    try:
        ea = EventAccumulator(str(event_file))
        ea.Reload()
        for tag in ea.Tags().get("scalars", []):
            if "loss" in tag.lower() or tag.lower() in {"lr", "learning_rate"}:
                for ev in ea.Scalars(tag):
                    tb_rows.append({
                        "tag": tag,
                        "step": ev.step,
                        "value": ev.value,
                        "event_file": str(event_file),
                    })
    except Exception as e:
        print("Could not read:", event_file, e)

tb_df = pd.DataFrame(tb_rows)
if len(tb_df):
    tb_df.to_csv(DRIVE_OUTPUT_ROOT / "tensorboard_scalars.csv", index=False)
    display(tb_df.head(20))
else:
    print("No scalar events were found. Open TensorBoard below to inspect the run.")


In [ ]:
# ============================================================
# 20. TRAINING LOSS CURVE
# ============================================================
if len(tb_df):
    loss_df = tb_df[tb_df["tag"].str.contains("loss", case=False, na=False)].copy()

    if len(loss_df):
        # If multiple event files/tags exist, plot all explicitly.
        fig, ax = plt.subplots(figsize=(10, 5))
        for tag, g in loss_df.groupby("tag"):
            g = g.sort_values("step")
            ax.plot(g["step"], g["value"], label=tag)
        ax.set_xlabel("Training step")
        ax.set_ylabel("Training loss")
        ax.set_title("FLUX.2 Klein LoRA training loss")
        ax.legend()
        ax.grid(True, alpha=0.25)
        plt.tight_layout()

        fig.savefig(DRIVE_OUTPUT_ROOT / "training_loss_curve.png", dpi=160)
        plt.show()

        print(
            "Important: a decreasing training loss does NOT by itself prove "
            "better held-out generation quality or attribute accuracy."
        )


## Training-process dashboard

The cells below use the **actual TensorBoard scalar events written by the trainer**. The moving-average curve is only a visualization aid; it does not replace the raw loss values.

In [ ]:
# ============================================================
# 20A. ADVANCED TRAINING CURVES
# ============================================================
if len(tb_df):
    scalar = tb_df.copy()
    scalar["step"] = pd.to_numeric(scalar["step"], errors="coerce")
    scalar["value"] = pd.to_numeric(scalar["value"], errors="coerce")
    scalar = scalar.dropna(subset=["step", "value"]).sort_values("step")

    # Identify the most useful loss and LR series.
    loss_candidates = scalar[scalar["tag"].str.contains("loss", case=False, na=False)]
    lr_candidates = scalar[
        scalar["tag"].str.contains(r"(^|/)(lr|learning_rate)$|learning.?rate",
                                   case=False, regex=True, na=False)
    ]

    if len(loss_candidates):
        loss_tag = loss_candidates["tag"].value_counts().index[0]
        train_loss = loss_candidates[loss_candidates["tag"] == loss_tag].copy()
        train_loss = train_loss.groupby("step", as_index=False)["value"].mean()
        train_loss["rolling_loss"] = train_loss["value"].rolling(
            max(3, min(25, len(train_loss) // 10)), min_periods=1
        ).mean()

        fig, ax = plt.subplots(figsize=(11, 5))
        ax.plot(train_loss["step"], train_loss["value"], alpha=0.30, label="Raw training loss")
        ax.plot(train_loss["step"], train_loss["rolling_loss"], linewidth=2, label="Moving average")
        ax.set_xlabel("Optimizer step")
        ax.set_ylabel("Training loss")
        ax.set_title("Training loss: raw and smoothed")
        ax.legend()
        ax.grid(True, alpha=0.25)
        plt.tight_layout()
        fig.savefig(DRIVE_OUTPUT_ROOT / "training_loss_raw_vs_smoothed.png", dpi=180)
        plt.show()

        # Summary statistics for the actual observed loss.
        best_row = train_loss.loc[train_loss["value"].idxmin()]
        print(f"Loss tag: {loss_tag}")
        print(f"First logged loss: {train_loss.iloc[0]['value']:.6f}")
        print(f"Final logged loss: {train_loss.iloc[-1]['value']:.6f}")
        print(f"Minimum logged loss: {best_row['value']:.6f} at step {int(best_row['step'])}")
        print(f"Relative change first→final: {(train_loss.iloc[-1]['value'] / train_loss.iloc[0]['value'] - 1) * 100:.2f}%")

        train_loss.to_csv(DRIVE_OUTPUT_ROOT / "training_loss_processed.csv", index=False)

    if len(lr_candidates):
        lr_tag = lr_candidates["tag"].value_counts().index[0]
        lr_df = lr_candidates[lr_candidates["tag"] == lr_tag].copy()
        lr_df = lr_df.groupby("step", as_index=False)["value"].mean()

        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(lr_df["step"], lr_df["value"])
        ax.set_xlabel("Optimizer step")
        ax.set_ylabel("Learning rate")
        ax.set_title("Learning-rate schedule")
        ax.grid(True, alpha=0.25)
        plt.tight_layout()
        fig.savefig(DRIVE_OUTPUT_ROOT / "learning_rate_curve.png", dpi=180)
        plt.show()

        lr_df.to_csv(DRIVE_OUTPUT_ROOT / "learning_rate_processed.csv", index=False)
else:
    print("No TensorBoard scalar data available. Run the TensorBoard cell after training.")


In [ ]:
# ============================================================
# 20B. TRAINING PROGRESS SUMMARY + CHECKPOINT TIMELINE
# ============================================================
# Estimate epoch position from the configured optimizer steps.
if "loss_df" in globals() and len(loss_df):
    progress = loss_df.groupby("step", as_index=False)["value"].mean().sort_values("step").copy()
    progress["estimated_epoch"] = progress["step"] / max(1, steps_per_epoch)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(progress["estimated_epoch"], progress["value"], alpha=0.75)
    ax.set_xlabel("Estimated epoch")
    ax.set_ylabel("Training loss")
    ax.set_title("Training loss by estimated epoch")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    fig.savefig(DRIVE_OUTPUT_ROOT / "training_loss_by_epoch.png", dpi=180)
    plt.show()

# Checkpoint timeline
if "checkpoint_inventory_df" in globals() and len(checkpoint_inventory_df):
    cp = checkpoint_inventory_df.sort_values("step").copy()
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.scatter(cp["step"], cp["size_gb"], s=70)
    for _, row in cp.iterrows():
        ax.annotate(
            row["checkpoint"],
            (row["step"], row["size_gb"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )
    ax.set_xlabel("Training step")
    ax.set_ylabel("Checkpoint size (GB)")
    ax.set_title("Checkpoint timeline and storage footprint")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    fig.savefig(DRIVE_OUTPUT_ROOT / "checkpoint_timeline.png", dpi=180)
    plt.show()

# Training configuration card as a simple research table.
config_rows = [
    ("Base model", MODEL_ID),
    ("Training resolution", TRAIN_RESOLUTION),
    ("Batch size", TRAIN_BATCH_SIZE),
    ("Gradient accumulation", GRAD_ACCUM_STEPS),
    ("Effective batch size", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS),
    ("LoRA rank", LORA_RANK),
    ("LoRA alpha", LORA_ALPHA),
    ("Learning rate", LEARNING_RATE),
    ("Warmup steps", WARMUP_STEPS),
    ("Target epochs", TARGET_EPOCHS),
    ("Maximum optimizer steps", MAX_TRAIN_STEPS_FINAL),
    ("Checkpoint interval", CHECKPOINT_STEPS),
    ("Validation images / epoch", NUM_VALIDATION_IMAGES),
]
config_table = pd.DataFrame(config_rows, columns=["Parameter", "Value"])
display(config_table)
config_table.to_csv(DRIVE_OUTPUT_ROOT / "training_configuration_table.csv", index=False)


### Validation-generation previews

The official trainer records validation **images**, not a held-out diffusion validation loss. The next cell tries to recover those images from TensorBoard event files and displays them chronologically when available.

In [ ]:
# ============================================================
# 20C. EXTRACT VALIDATION IMAGES FROM TENSORBOARD
# ============================================================
from io import BytesIO

validation_gallery = []

for event_file in event_files:
    try:
        ea = EventAccumulator(str(event_file))
        ea.Reload()
        image_tags = ea.Tags().get("images", [])
        for tag in image_tags:
            # Prefer validation-related image summaries.
            if "validation" not in tag.lower() and "val" not in tag.lower():
                continue
            for ev in ea.Images(tag):
                try:
                    img = Image.open(BytesIO(ev.encoded_image_string)).convert("RGB")
                    validation_gallery.append({
                        "tag": tag,
                        "step": int(ev.step),
                        "width": int(ev.width),
                        "height": int(ev.height),
                        "image": img.copy(),
                    })
                finally:
                    try:
                        img.close()
                    except Exception:
                        pass
    except Exception as e:
        print("Could not inspect validation images in:", event_file, e)

if validation_gallery:
    validation_gallery = sorted(validation_gallery, key=lambda x: (x["step"], x["tag"]))
    gallery_dir = DRIVE_OUTPUT_ROOT / "validation_gallery"
    gallery_dir.mkdir(parents=True, exist_ok=True)

    for i, item in enumerate(validation_gallery):
        out = gallery_dir / f"step_{item['step']:07d}_{i:03d}.png"
        item["image"].save(out)
        item["path"] = str(out)

    display_n = min(12, len(validation_gallery))
    cols = 4
    rows = math.ceil(display_n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, item in zip(axes, validation_gallery[:display_n]):
        ax.imshow(item["image"])
        ax.set_title(f"{item['tag']} | step {item['step']}")
        ax.axis("off")
    for ax in axes[display_n:]:
        ax.axis("off")

    plt.suptitle("Validation generations recorded by the trainer", y=1.01, fontsize=14)
    plt.tight_layout()
    fig.savefig(DRIVE_OUTPUT_ROOT / "validation_generation_gallery.png", dpi=180, bbox_inches="tight")
    plt.show()

    validation_manifest = pd.DataFrame([
        {k: v for k, v in item.items() if k != "image"}
        for item in validation_gallery
    ])
    validation_manifest.to_csv(DRIVE_OUTPUT_ROOT / "validation_generation_manifest.csv", index=False)
    print(f"Recovered {len(validation_gallery)} validation images.")
else:
    print(
        "No validation image summaries were found in TensorBoard event files. "
        "This does not mean validation generation failed; inspect TensorBoard "
        "or the trainer output directory for validation artifacts."
    )


### How validation is recorded

During the training cell, `--validation_prompt`, `--validation_epochs=1`, and `--num_validation_images=2` are passed to the official Klein trainer. That produces validation generations after each epoch and logs them to TensorBoard. This is the trainer's supported validation mechanism. citeturn0search0turn0search1

For the thesis, treat these validation generations as **qualitative/visual validation evidence**, while the held-out **test** split remains the quantitative evaluation set.


In [ ]:
# ============================================================
# 21. SAVE THE VALIDATION PROTOCOL
# ============================================================
validation_protocol = {
    "validation_split": "validation",
    "validation_loss": None,
    "validation_loss_reason": "Official Klein LoRA trainer does not expose a held-out diffusion validation-loss loop.",
    "validation_generation": True,
    "validation_epochs": VALIDATION_EPOCHS,
    "validation_images_per_epoch": NUM_VALIDATION_IMAGES,
    "validation_prompt": "A professional high-end studio product photograph of a luxury sapphire jewelry piece, extremely detailed gemstone and metal, clean neutral background",
    "training_loss_source": "TensorBoard event files",
    "quantitative_final_evaluation_split": "test",
}
with (DRIVE_OUTPUT_ROOT / "validation_protocol.json").open("w", encoding="utf-8") as f:
    json.dump(validation_protocol, f, indent=2)
print(json.dumps(validation_protocol, indent=2))


### Resume after a Colab disconnect

If the runtime disconnects after a checkpoint has been saved, reconnect the notebook, rerun the setup/data cells, and set:

```python
RESUME_FROM_CHECKPOINT = "latest"
```

Then rerun the training-command and training cells. The official trainer supports `--resume_from_checkpoint latest` and checkpoint paths for this purpose. citeturn6view0


In [ ]:
# ============================================================
# 22. OPEN TENSORBOARD IN COLAB
# ============================================================
# Run this after training. It displays the training loss logged by the
# official trainer. It may take a few seconds to discover the event file.

%load_ext tensorboard
%tensorboard --logdir "$OUTPUT_DIR" --port 6006


In [ ]:
# ============================================================
# 23. FIND FINAL LORA
# ============================================================
all_safetensors = sorted(OUTPUT_DIR.rglob("*.safetensors"), key=lambda p: p.stat().st_mtime)
for p in all_safetensors[-20:]:
    print(p)

if not all_safetensors:
    raise FileNotFoundError("No .safetensors LoRA file found.")

preferred = [p for p in all_safetensors if p.name == "pytorch_lora_weights.safetensors"]
FINAL_LORA = preferred[-1] if preferred else all_safetensors[-1]
print("Selected:", FINAL_LORA)

# Stage 3: held-out benchmark

We create a fixed benchmark from the **test** split. The base and fine-tuned images use the exact same prompts and exact same seeds. This makes visual and metric comparisons much more meaningful than comparing unrelated generations.

In [ ]:
# ============================================================
# 24. SELECT TEST BENCHMARK
# ============================================================
groups = [c for c in ["category", "gemstone", "metal"] if c in test_df.columns]
parts = []
if groups:
    target_per_group = max(1, BENCHMARK_COUNT // max(1, test_df.groupby(groups).ngroups))
    for _, g in test_df.groupby(groups, dropna=False):
        parts.append(g.sample(min(len(g), target_per_group), random_state=SEED))
benchmark_df = pd.concat(parts, ignore_index=True) if parts else test_df.copy()

if len(benchmark_df) < BENCHMARK_COUNT:
    remaining = test_df[~test_df["image_id"].isin(benchmark_df["image_id"])]
    extra = remaining.sample(min(BENCHMARK_COUNT-len(benchmark_df), len(remaining)), random_state=SEED)
    benchmark_df = pd.concat([benchmark_df, extra], ignore_index=True)

benchmark_df = benchmark_df.sample(min(BENCHMARK_COUNT, len(benchmark_df)), random_state=SEED).reset_index(drop=True)
benchmark_df["benchmark_seed"] = [SEED + 17*i for i in range(len(benchmark_df))]
benchmark_df.to_csv(DRIVE_OUTPUT_ROOT / "benchmark_prompts.csv", index=False)

display(benchmark_df[["image_id","category","gemstone","gemstone_color","gemstone_cut","metal","setting","style","benchmark_seed"]])

In [ ]:
# ============================================================
# 25. LOAD BASE PIPELINE FOR PAIRED GENERATION
# ============================================================
import torch
from diffusers import DiffusionPipeline

dtype = torch.bfloat16 if USE_BF16 else torch.float16
BASE_GENERATION_ROOT = DRIVE_OUTPUT_ROOT / "benchmark" / "base"
LORA_GENERATION_ROOT = DRIVE_OUTPUT_ROOT / "benchmark" / "finetuned"
BASE_GENERATION_ROOT.mkdir(parents=True, exist_ok=True)
LORA_GENERATION_ROOT.mkdir(parents=True, exist_ok=True)

pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe.enable_model_cpu_offload()
print("Pipeline loaded.")

In [ ]:
# ============================================================
# 26. GENERATE BASE + FINE-TUNED IMAGES WITH SAME SEEDS
# ============================================================
manifest = []

for i, row in benchmark_df.iterrows():
    prompt = row["structured_prompt"]
    seed = int(row["benchmark_seed"])

    gen = torch.Generator(device="cpu").manual_seed(seed)
    base_img = pipe(
        prompt=prompt,
        width=1024,
        height=1024,
        num_inference_steps=INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=gen,
    ).images[0]
    base_path = BASE_GENERATION_ROOT / f"{i:03d}_{row['image_id']}_base.png"
    base_img.save(base_path)

    pipe.load_lora_weights(
        str(FINAL_LORA.parent),
        weight_name=FINAL_LORA.name,
        adapter_name="jewelry"
    )
    try:
        pipe.set_adapters("jewelry", LORA_SCALE)
    except Exception:
        pass

    gen = torch.Generator(device="cpu").manual_seed(seed)
    lora_img = pipe(
        prompt=prompt,
        width=1024,
        height=1024,
        num_inference_steps=INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=gen,
    ).images[0]
    lora_path = LORA_GENERATION_ROOT / f"{i:03d}_{row['image_id']}_finetuned.png"
    lora_img.save(lora_path)

    try:
        pipe.unload_lora_weights()
    except Exception:
        pass

    manifest.append({
        "image_id": row["image_id"],
        "seed": seed,
        "prompt": prompt,
        "base_path": str(base_path),
        "finetuned_path": str(lora_path),
    })
    print(f"[{i+1}/{len(benchmark_df)}] {row['image_id']}")

with (DRIVE_OUTPUT_ROOT / "benchmark_manifest.json").open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Benchmark generation complete.")

In [ ]:
# ============================================================
# 27. VISUAL BASE VS FINETUNED GRID
# ============================================================
def compare_grid(items, n=8):
    items = items[:n]
    fig, axes = plt.subplots(len(items), 2, figsize=(12, 5*len(items)))
    if len(items) == 1:
        axes = np.array([axes])
    for r, item in enumerate(items):
        axes[r,0].imshow(Image.open(item["base_path"]).convert("RGB"))
        axes[r,0].set_title(f"BASE — {item['image_id']}")
        axes[r,0].axis("off")
        axes[r,1].imshow(Image.open(item["finetuned_path"]).convert("RGB"))
        axes[r,1].set_title(f"FINE-TUNED — {item['image_id']}")
        axes[r,1].axis("off")
    plt.tight_layout()
    plt.show()

compare_grid(manifest, min(8, len(manifest)))

# Stage 4: semantic prompt alignment

This metric is useful, but it is **not the final correctness metric**. A visually plausible sapphire ring can still have the wrong cut or setting. That is why the next stage scores individual attributes.

In [ ]:
# ============================================================
# 28. OPTIONAL SIGLIP2 ALIGNMENT
# ============================================================
if RUN_SIGLIP:
    from transformers import AutoProcessor, AutoModel
    siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_ID)
    siglip_model = AutoModel.from_pretrained(SIGLIP_MODEL_ID, torch_dtype=dtype, device_map="auto")
    siglip_device = next(p for p in siglip_model.parameters() if p.device.type != "meta").device

    def siglip_score(image_path, prompt):
        image = Image.open(image_path).convert("RGB")
        inputs = siglip_processor(text=[prompt], images=image, padding="max_length", return_tensors="pt")
        inputs = {k:(v.to(siglip_device) if hasattr(v,"to") else v) for k,v in inputs.items()}
        with torch.no_grad():
            out = siglip_model(**inputs)
        a, b = out.image_embeds, out.text_embeds
        a = a / a.norm(dim=-1, keepdim=True)
        b = b / b.norm(dim=-1, keepdim=True)
        return float((a*b).sum().item())

    rows = []
    for item in manifest:
        rows.append({
            "image_id": item["image_id"],
            "seed": item["seed"],
            "base_alignment": siglip_score(item["base_path"], item["prompt"]),
            "finetuned_alignment": siglip_score(item["finetuned_path"], item["prompt"]),
        })
    align_df = pd.DataFrame(rows)
    align_df["improvement"] = align_df["finetuned_alignment"] - align_df["base_alignment"]
    display(align_df)
    align_df.to_csv(DRIVE_OUTPUT_ROOT / "siglip_alignment.csv", index=False)
    print("Mean base:", align_df["base_alignment"].mean())
    print("Mean fine-tuned:", align_df["finetuned_alignment"].mean())

    # Free memory before loading the VLM evaluator.
    del siglip_model, siglip_processor
    gc.collect()
    torch.cuda.empty_cache()

# Stage 5: attribute correctness with a vision-language evaluator

The VLM is asked to compare the generated image with the expected metadata. It returns `1` for clearly correct, `0` for clearly wrong, and `-1` for not visually determinable.

**Exact carat weight is intentionally excluded from the image correctness score** because it usually cannot be established from a product image alone.

In [ ]:
# ============================================================
# 29. OPTIONAL QWEN3-VL ATTRIBUTE EVALUATION
# ============================================================
if RUN_VLM_ATTRIBUTE_EVAL:
    from transformers import pipeline
    evaluator = pipeline("image-text-to-text", model=VLM_MODEL_ID, device_map="auto", torch_dtype=dtype)
    lookup = test_df.set_index("image_id")

    attrs = [
        "category","gemstone","gemstone_color","gemstone_cut",
        "metal","setting","style","occasion"
    ]

    def eval_prompt(row):
        target = {a: clean_text(row[a]) for a in attrs}
        return f"""
Evaluate this jewelry image against the target metadata. Inspect only visible evidence.
Use 1 if clearly correct, 0 if clearly incorrect, and -1 if the attribute cannot be determined reliably.
Do not guess hidden properties such as exact carat weight.

Target metadata:
{json.dumps(target, indent=2)}

Return ONLY valid JSON:
{{
  "category": 1,
  "gemstone": 1,
  "gemstone_color": 1,
  "gemstone_cut": 1,
  "metal": 1,
  "setting": 1,
  "style": 1,
  "occasion": 1,
  "visual_quality_1_to_5": 1,
  "brief_reason": "..."
}}
""".strip()

    out_rows = []
    for item in manifest:
        target = lookup.loc[item["image_id"]]
        for model_name, image_path in [("base", item["base_path"]), ("finetuned", item["finetuned_path"])]:
            messages = [{
                "role":"user",
                "content":[
                    {"type":"image", "url":str(image_path)},
                    {"type":"text", "text":eval_prompt(target)},
                ]
            }]
            result = evaluator(text=messages, max_new_tokens=300, return_full_text=False)[0]["generated_text"]
            m = re.search(r"\{.*\}", result, flags=re.DOTALL)
            try:
                parsed = json.loads(m.group(0)) if m else {}
            except Exception:
                parsed = {}
            for a in attrs:
                if parsed.get(a) not in [0,1,-1]:
                    parsed[a] = -1
            try:
                q = float(parsed.get("visual_quality_1_to_5", -1))
            except Exception:
                q = -1
            parsed["visual_quality_1_to_5"] = q
            vals = [parsed[a] for a in attrs if parsed[a] in [0,1]]
            parsed["attribute_accuracy"] = float(np.mean(vals)) if vals else np.nan
            parsed["image_id"] = item["image_id"]
            parsed["model"] = model_name
            parsed["raw_output"] = result[:2000]
            out_rows.append(parsed)

    attr_df = pd.DataFrame(out_rows)
    attr_df.to_csv(DRIVE_OUTPUT_ROOT / "attribute_evaluation.csv", index=False)
    display(attr_df[["image_id","model","attribute_accuracy","visual_quality_1_to_5","brief_reason"]])
    print("\nMean attribute accuracy:")
    display(attr_df.groupby("model")["attribute_accuracy"].mean().to_frame())

## Optional FID

FID is included only as a secondary distribution metric. It should not replace attribute accuracy or human evaluation for jewelry because a lower FID does not necessarily mean a better match to the requested metal, gemstone cut, or setting.

In [ ]:
# ============================================================
# 30. OPTIONAL FID
# ============================================================
if RUN_FID:
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchvision import transforms

    fid_base = FrechetInceptionDistance(feature=2048, normalize=False).to("cuda")
    fid_lora = FrechetInceptionDistance(feature=2048, normalize=False).to("cuda")
    to_tensor = transforms.Compose([
        transforms.Resize((299,299)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: (x*255).to(torch.uint8)),
    ])

    real_paths = test_df.sample(min(len(test_df), len(manifest)), random_state=SEED)["resolved_path"].tolist()
    for p in real_paths:
        x = to_tensor(Image.open(p).convert("RGB")).unsqueeze(0).cuda()
        fid_base.update(x, real=True); fid_lora.update(x, real=True)
    for item in manifest:
        xb = to_tensor(Image.open(item["base_path"]).convert("RGB")).unsqueeze(0).cuda()
        xl = to_tensor(Image.open(item["finetuned_path"]).convert("RGB")).unsqueeze(0).cuda()
        fid_base.update(xb, real=False); fid_lora.update(xl, real=False)
    print("FID base:", float(fid_base.compute()))
    print("FID fine-tuned:", float(fid_lora.compute()))

In [ ]:
# ============================================================
# 31. FINAL EXPERIMENT SUMMARY
# ============================================================
summary = {
    "model": MODEL_ID,
    "train_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "benchmark_images": int(len(manifest)),
    "train_resolution": TRAIN_RESOLUTION,
    "lora_rank": LORA_RANK,
    "learning_rate": LEARNING_RATE,
    "max_train_steps": MAX_TRAIN_STEPS_FINAL,
    "checkpoint_steps": CHECKPOINT_STEPS,
    "fp8_training": FP8_TRAINING,
    "mixed_precision": MIXED_PRECISION,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "official_cache": False,
    "official_cache_root": str(OUTPUT_DIR),
    "checkpoints_to_keep": CHECKPOINTS_TO_KEEP,
    "validation_loss": "Not computed by the official Klein trainer; validation is generation-based.",
}

if "align_df" in globals():
    summary["mean_siglip_base"] = float(align_df["base_alignment"].mean())
    summary["mean_siglip_finetuned"] = float(align_df["finetuned_alignment"].mean())
if "attr_df" in globals():
    summary["mean_attribute_accuracy"] = {str(k): float(v) for k,v in attr_df.groupby("model")["attribute_accuracy"].mean().to_dict().items()}

with (DRIVE_OUTPUT_ROOT / "final_experiment_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nAll experiment outputs:", DRIVE_OUTPUT_ROOT)

## Final evaluation visualizations

These plots summarize the paired base-vs-fine-tuned benchmark. They use the metrics already computed by the notebook and do not create additional evaluation data.

In [ ]:
# ============================================================
# 31A. FINAL METRIC VISUALIZATION
# ============================================================

# Semantic alignment comparison
if "align_df" in globals() and len(align_df):
    means = align_df[["base_alignment", "finetuned_alignment"]].mean()
    labels = ["Base", "Fine-tuned"]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, [means["base_alignment"], means["finetuned_alignment"]])
    ax.set_ylabel("Mean SigLIP image-text alignment")
    ax.set_title("Semantic prompt alignment")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    fig.savefig(DRIVE_OUTPUT_ROOT / "final_siglip_comparison.png", dpi=180)
    plt.show()

# Attribute accuracy comparison
if "attr_df" in globals() and len(attr_df):
    attr_means = attr_df.groupby("model")["attribute_accuracy"].mean().reindex(
        ["base", "finetuned"]
    )

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(["Base", "Fine-tuned"], attr_means.values)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Mean attribute accuracy")
    ax.set_title("Visible attribute correctness")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    fig.savefig(DRIVE_OUTPUT_ROOT / "final_attribute_accuracy_comparison.png", dpi=180)
    plt.show()

# Per-attribute comparison when VLM results are available.
if "attr_df" in globals() and len(attr_df):
    available_attrs = [a for a in attrs if a in attr_df.columns]
    per_attr_rows = []
    for a in available_attrs:
        for model_name in ["base", "finetuned"]:
            valid = attr_df.loc[attr_df["model"] == model_name, a]
            valid = valid[valid.isin([0, 1])]
            per_attr_rows.append({
                "attribute": a,
                "model": model_name,
                "accuracy": valid.mean() if len(valid) else np.nan,
            })
    per_attr_df = pd.DataFrame(per_attr_rows)

    fig, ax = plt.subplots(figsize=(12, 5))
    pivot = per_attr_df.pivot(index="attribute", columns="model", values="accuracy")
    pivot.reindex(columns=["base", "finetuned"]).plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_xlabel("")
    ax.set_title("Attribute-level correctness: base vs fine-tuned")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    fig.savefig(DRIVE_OUTPUT_ROOT / "attribute_level_comparison.png", dpi=180)
    plt.show()

    per_attr_df.to_csv(DRIVE_OUTPUT_ROOT / "attribute_accuracy_by_field.csv", index=False)

# Combined research summary table
research_rows = []
if "align_df" in globals() and len(align_df):
    research_rows.extend([
        {"metric": "Mean SigLIP alignment", "base": align_df["base_alignment"].mean(),
         "fine_tuned": align_df["finetuned_alignment"].mean()},
    ])
if "attr_df" in globals() and len(attr_df):
    research_rows.extend([
        {"metric": "Mean attribute accuracy", "base": attr_df.loc[attr_df["model"]=="base", "attribute_accuracy"].mean(),
         "fine_tuned": attr_df.loc[attr_df["model"]=="finetuned", "attribute_accuracy"].mean()},
        {"metric": "Mean visual quality (1–5)", "base": attr_df.loc[attr_df["model"]=="base", "visual_quality_1_to_5"].replace(-1, np.nan).mean(),
         "fine_tuned": attr_df.loc[attr_df["model"]=="finetuned", "visual_quality_1_to_5"].replace(-1, np.nan).mean()},
    ])

research_summary_df = pd.DataFrame(research_rows)
if len(research_summary_df):
    display(research_summary_df.round(4))
    research_summary_df.to_csv(DRIVE_OUTPUT_ROOT / "research_metric_summary.csv", index=False)


# Recommended research interpretation

Do not select a checkpoint solely from the lowest training loss.

For this task, the strongest model should balance:

- visual realism and jewelry detail
- category correctness
- gemstone correctness
- gemstone color correctness
- cut correctness
- metal correctness
- setting correctness
- style/occasion consistency
- semantic prompt alignment
- low memorization of training images

For the final report, use a table such as:

| Metric | Base | Fine-tuned |
|---|---:|---:|
| Mean semantic alignment | … | … |
| Attribute accuracy | … | … |
| Human visual quality | … | … |
| Optional FID | … | … |

The automated VLM evaluator is a proxy, not perfect ground truth. A blinded human review is strongly recommended for subtle properties such as gemstone cut and metal/setting details.

## References used for the implementation

- Black Forest Labs FLUX.2 repository: https://github.com/black-forest-labs/flux2
- Hugging Face Diffusers FLUX.2 training guide: https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/README_flux2.md
- FLUX.2 Klein 9B Base model: https://huggingface.co/black-forest-labs/FLUX.2-klein-base-9B
- Qwen3-VL-4B-Instruct: https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct